# 📊 实验 23：工具调用综合实验（FC vs MCP vs CLI vs Agent 效率与 Token 消耗对比）

### 💡 实验设想与背景
在设计 AI Agent 系统时，我们需要在不同的“工具抽象级别”与“智能体架构”之间做出选择。本实验比较四种主流工具调用范式在不同目录深度下的表现：
1. **CLI 命令行检索（粗粒度/单次）**：宿主直接执行递归查找，大模型只需 1 轮工具调用便可定位任意深度的文件。
2. **Function Calling 细粒度检索（多轮 ReAct）**：模型仅能使用 `list_directory` 列出单层目录，必须逐级下探。随着目录深度增加，交互轮数线性增加，Token 呈二次方增长。
3. **MCP 细粒度检索（多轮 MCP）**：通过 Stdio 连接标准的 MCP 文件服务器，模型使用 MCP 提供的细粒度接口逐层下探。
4. **层级代理 (Hierarchical Agent)**：项目经理 (Manager) 负责顶层指挥与最终总结，具体寻找文件的繁重工作通过工具外包给专员 (Worker)。专员拥有细粒度工具，逐级查找。

### 🔬 实验设计
我们在 `data/fc_test` 目录下自动构建一个 **10 级深层目录树**。在每一级目录中放置一个对应深度的密钥文件 `secret_flag_level_{depth}.txt`。接着，我们将对于深度 1 至 10，分别运行上述四种方式进行查找，记录并对比它们的：
- **总耗时 (Response Time)**
- **工具调用轮数 (Turns)**
- **输入/输出 Token 消耗 (Prompt / Completion Tokens)**
- 对于层级代理 (Agent)，我们将**经理 (Manager)**和**专员 (Worker)**的消耗指标分开统计并展示。

### 🛠️ 1. 环境准备与客户端初始化

导入项目依赖的 Python 标准库与第三方库，配置 API Key、Base URL 以及模型名称，并实例化 OpenAI 客户端。


In [ ]:
import os
import json
import time
import sys
import httpx
from openai import OpenAI

# ============================================================
# 👇 请在下方引号内填入您的 API Key（也可以直接读取系统环境变量）
# ============================================================
API_KEY = ""         # 例如: "sk-abc123..."
BASE_URL = "https://api.siliconflow.cn/v1"        # 例如: "https://api.siliconflow.cn/v1" 或 "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "deepseek-ai/DeepSeek-V3"      # 例如: "deepseek-ai/DeepSeek-V3" 或 "qwen-plus"

# 优先从上面填写的变量读取，其次读取系统环境变量
api_key = API_KEY or os.environ.get("OPENAI_API_KEY") or os.environ.get("SILICONFLOW_API_KEY") or os.environ.get("DASH_SCOPE_API_KEY")
base_url = BASE_URL or os.environ.get("OPENAI_API_BASE")
model_name = MODEL_NAME or os.environ.get("OPENAI_API_MODEL")

# 自动识别常见服务商的环境变量
if not base_url:
    if os.environ.get("SILICONFLOW_API_KEY"):
        base_url = "https://api.siliconflow.cn/v1"
        model_name = model_name or "deepseek-ai/DeepSeek-V3"
    elif os.environ.get("DASH_SCOPE_API_KEY"):
        base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"
        model_name = model_name or "qwen-plus"
    else:
        base_url = "https://api.openai.com/v1"
        model_name = model_name or "gpt-4o-mini"

if not api_key:
    raise ValueError("❌ 未检测到 API Key！请在上方单元格中配置您的 API Key 或设置系统环境变量。")

openai_client = OpenAI(api_key=api_key, base_url=base_url)
print(f"✨ 客户端实例化成功！当前使用的接口地址为: {base_url}，模型为: {model_name}")

### 📁 2. 构建测试用多级目录结构

定义辅助函数 `setup_complex_directory()`，在 `data/fc_test` 下自动构建一个 **10 级深层的目录树**。并在每一级目录中随机生成 1 至 2 个文件夹，以测试不同深度与分支结构下的文件查找效率。


In [ ]:
import shutil
import random

def setup_complex_directory():
    """
    在 data/fc_test 下生成一个 10 级深层目录树。
    在每个深度写入一个密钥文件 secret_flag_level_{depth}.txt。
    每一级随机存在 1-2 个文件夹。
    """
    base_dir = "data/fc_test"
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    os.makedirs(base_dir, exist_ok=True)
    
    current_path = base_dir
    level_files = {}
    
    for depth in range(1, 11):
        # 决定该层级的文件夹数量：随机 1 到 2 个
        # 必含一个指向更深层次的 active folder，可能包含一个干扰用的 dummy folder
        num_folders = random.randint(1, 2)
        
        # 1. 创建指向下一级的 active folder
        active_folder_name = f"dir_level_{depth}"
        active_folder_path = os.path.join(current_path, active_folder_name)
        os.makedirs(active_folder_path, exist_ok=True)
        
        # 2. 如果是 2 个文件夹，则额外创建一个 dummy folder
        if num_folders == 2:
            dummy_folder_name = f"dummy_dir_level_{depth}"
            dummy_folder_path = os.path.join(current_path, dummy_folder_name)
            os.makedirs(dummy_folder_path, exist_ok=True)
            # 在 dummy folder 中写入一些干扰文件
            with open(os.path.join(dummy_folder_path, f"dummy_note_{depth}.txt"), "w", encoding="utf-8") as f:
                f.write(f"This is a dummy path at depth {depth}.")
        
        # 将 current_path 更新为下一级 active folder，并在其下写入此深度的目标文件和干扰文件
        current_path = active_folder_path
        
        # 写入此深度的目标文件
        target_name = f"secret_flag_level_{depth}.txt"
        target_path = os.path.join(current_path, target_name)
        with open(target_path, "w", encoding="utf-8") as f:
            f.write(f"SUCCESS: KEY=LEVEL_{depth}_SECRET_KEY_9988")
        level_files[depth] = (target_name, os.path.abspath(target_path))
        
        # 写入几个干扰文件
        for i in range(2):
            dummy_name = f"dummy_level_{depth}_{i}.txt"
            with open(os.path.join(current_path, dummy_name), "w", encoding="utf-8") as f:
                f.write(f"Dummy file at level {depth}")
                
    return level_files

level_files = setup_complex_directory()
print("🎯 10 级包含分支的深层目录树构建完毕，密钥文件生成就绪！")

### 🔄 3. 异步执行辅助器

定义 `run_async` 函数，在新线程中启动独立的 asyncio 事件循环。这能够有效避免在 Jupyter Notebook 现有的事件循环中直接运行异步任务时抛出错误。


In [ ]:
import threading
import asyncio

def run_async(coro):
    """
    在新线程中运行一个独立的 asyncio 事件循环，确保在 Jupyter Notebook 中不会因为 event loop 已在运行而抛出异常。
    """
    result = []
    error = []
    def target():
        try:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            res = loop.run_until_complete(coro)
            result.append(res)
        except Exception as e:
            error.append(e)
        finally:
            loop.close()
    thread = threading.Thread(target=target)
    thread.start()
    thread.join()
    if error:
        raise error[0]
    return result[0]
print("✅ run_async 异步执行器加载完成。")

### 🔌 4. 范式一：细粒度 Function Calling 实验

定义细粒度的文件及文件夹操作工具（`list_directory` 和 `read_file`）。在此模式下，大模型没有递归查找工具，必须通过多轮 ReAct 交互，逐级列出目录并下探，直到找到目标文件。


In [ ]:
def list_directory(path: str) -> dict:
    """
    列出相对指定目录下的下一级子目录和文件。非递归。
    """
    try:
        if not os.path.exists(path):
            return {"error": f"Path {path} does not exist."}
        dirs = []
        files = []
        for item in os.listdir(path):
            full_p = os.path.join(path, item)
            if os.path.isdir(full_p):
                dirs.append(item)
            else:
                files.append(item)
        return {"directories": dirs, "files": files}
    except Exception as e:
        return {"error": str(e)}

def read_file(path: str) -> str:
    """
    读取指定文件的内容
    """
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        return f"[ERROR] Cannot read file {path}: {str(e)}"

# ============= 🧠 TODO 任务 1：补全 Function Calling 的 Tool Schema =================
fine_grained_tools = [
    {
        "type": "",
        "function": {
            "name": "",
            "description": "",
            "parameters": {
                "type": "",
                "properties": {
                    
                },
                "required": []
            }
        }
    },
    {
        "type": "",
        "function": {
            "name": "",
            "description": "",
            "parameters": {
                "type": "",
                "properties": {
                    
                },
                "required": []
            }
        }
    }
]
# ==================================================================================

def run_fc_experiment(target_filename):
    system_prompt = (
        "你是一个深层目录文件搜索专员。\n"
        "你需要逐步查找目标文件，注意你只能使用 list_directory 一级一级探索，不能直接猜测或递归。\n"
        "请每次列出当前目录，如果在 files 中找到目标文件，立即调用 read_file 工具读取其内容并回答。\n"
        "如果看到子目录，就立刻进入下一级目录。请一步步探索直至找到目标文件。"
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"请在目录 data/fc_test 的目录树中，一步步搜寻文件 '{target_filename}' 的具体内容。第一步请先列出 'data/fc_test'。"}
    ]
    
    p_tokens, c_tokens, turns = 0, 0, 0
    start_time = time.time()
    
    while turns < 20:  # 工具调用轮数限制最大 20 轮
        response = openai_client.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=fine_grained_tools,
            tool_choice="auto"
        )
        
        p_tokens += response.usage.prompt_tokens
        c_tokens += response.usage.completion_tokens
        turns += 1
        
        assistant_msg = response.choices[0].message
        messages.append(assistant_msg)
        
        if assistant_msg.tool_calls:
            for call in assistant_msg.tool_calls:
                func_name = call.function.name
                func_args = json.loads(call.function.arguments)
                
                if func_name == "list_directory":
                    res = str(list_directory(func_args.get("path")))
                elif func_name == "read_file":
                    res = read_file(func_args.get("path"))
                else:
                    res = "Unknown tool"
                    
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "name": func_name,
                    "content": res
                })
        else:
            break
            
    elapsed = time.time() - start_time
    
    import re
    match = re.search(r'level_(\d+)', target_filename)
    depth = int(match.group(1)) if match else None
    expected_key = f"LEVEL_{depth}_SECRET_KEY_9988" if depth is not None else ""
    
    found_key = False
    for msg in messages:
        content = ""
        if hasattr(msg, 'content') and msg.content:
            content = msg.content
        elif isinstance(msg, dict):
            content = msg.get('content', '')
        
        if content and expected_key in content:
            found_key = True
            break
            
    return {
        "time": elapsed,
        "turns": turns,
        "prompt_tokens": p_tokens,
        "completion_tokens": c_tokens,
        "total_tokens": p_tokens + c_tokens,
        "success": found_key,
        "key": expected_key if found_key else "未找到"
    }
print("✅ Function Calling 实验模块加载成功。")

#### 🔌 运行 Function Calling 评测测试

执行深度 1 至 10 的 Function Calling 文件查找，记录并存储每一步的耗时、轮数和 Token 消耗。


In [ ]:
fc_results = []
print("🚀 开始进行细粒度 Function Calling 实验 (深度 1 至 10)...")
for depth in range(1, 11):
    target_name, target_path = level_files[depth]
    print(f"   [FC 测试] 正在查找深度 {depth}: {target_name}...")
    res = run_fc_experiment(target_name)
    fc_results.append({
        "depth": depth,
        "time": res["time"],
        "turns": res["turns"],
        "tokens": res["total_tokens"]
    })
    print(f"            耗时: {res['time']:.2f}s | 轮数: {res['turns']} | 总Token: {res['total_tokens']} | 结果: {res.get('key', '未找到')}")
print("✅ Function Calling 实验测试完成！")

### 💻 5. 范式二：CLI 命令行检索实验

像真实 Agent 调用 CLI 终端一样，本节不为特定查找任务编写专用的 Python 函数包装器，而是注册一个通用的受控终端命令执行器 `run_command`。白名单内仅允许调用安全命令（如 `pwd`、`ls`、`grep`、`cat`、`echo`），具体如何利用这些命令检索并读取目标文件，完全交由大模型自行推理和组合。


In [ ]:
import subprocess
import sys
import os
import shlex

# 定义安全命令白名单
ALLOWED_COMMANDS = {"pwd", "ls", "grep", "cat", "echo"}

def run_command(command: str) -> str:
    """
    通用命令执行器：执行终端命令，但仅限安全白名单命令，支持跨平台仿真。
    """
    try:
        # 1. 安全解析命令行参数，避免 Shell 注入
        args = shlex.split(command)
        if not args:
            return "错误：命令不能为空。"
        
        base_cmd = args[0]
        
        # 2. 白名单检查
        if base_cmd not in ALLOWED_COMMANDS:
            return f"⚠️ 安全警告：非法的命令调用！该环境仅允许运行以下安全命令: {list(ALLOWED_COMMANDS)}"
        
        # 3. 针对 Windows 环境进行常用命令仿真，防止 executable 找不到报错
        if os.name == 'nt':
            if base_cmd == "pwd":
                return os.getcwd()
            elif base_cmd == "ls":
                recursive = False
                target_path = "."
                for arg in args[1:]:
                    if arg.startswith("-"):
                        if "R" in arg:
                            recursive = True
                    else:
                        target_path = arg
                
                if os.path.exists(target_path):
                    if recursive:
                        result_lines = []
                        for root, dirs, files in os.walk(target_path):
                            rel_path = os.path.relpath(root, target_path)
                            prefix = "" if rel_path == "." else rel_path + "/"
                            for d in dirs:
                                result_lines.append(prefix + d + "/")
                            for f in files:
                                result_lines.append(prefix + f)
                        return "\n".join(result_lines)
                    else:
                        return "\n".join(os.listdir(target_path))
                return f"ls: {target_path}: No such file or directory"
            elif base_cmd == "cat":
                if len(args) < 2:
                    return "cat: missing operand"
                path = args[1]
                if os.path.exists(path):
                    with open(path, 'r', encoding='utf-8') as f:
                        return f.read()
                return f"cat: {path}: No such file or directory"
            elif base_cmd == "echo":
                return " ".join(args[1:])
            elif base_cmd == "grep":
                recursive = False
                pattern = None
                target_path = None
                
                options = []
                remaining_args = []
                for arg in args[1:]:
                    if arg.startswith("-"):
                        options.append(arg)
                        if "r" in arg or "R" in arg:
                            recursive = True
                    else:
                        remaining_args.append(arg)
                
                if len(remaining_args) >= 2:
                    pattern = remaining_args[0]
                    target_path = remaining_args[1]
                elif len(remaining_args) == 1:
                    pattern = remaining_args[0]
                    target_path = "."
                else:
                    return "grep: format is 'grep <pattern> <file>' or 'grep -r <pattern> <dir>'"
                
                if os.path.exists(target_path):
                    if os.path.isdir(target_path):
                        if recursive:
                            matched_lines = []
                            for root, dirs, files in os.walk(target_path):
                                for file in files:
                                    filepath = os.path.join(root, file)
                                    try:
                                        with open(filepath, 'r', encoding='utf-8') as f:
                                            for line_num, line in enumerate(f, 1):
                                                if pattern in line:
                                                    rel_path = os.path.relpath(filepath)
                                                    matched_lines.append(f"{rel_path}:{line_num}:{line.strip()}")
                                    except Exception:
                                        pass
                            return "\n".join(matched_lines)
                        else:
                            return f"grep: {target_path}: Is a directory (use -r to search recursively)"
                    else:
                        with open(target_path, 'r', encoding='utf-8') as f:
                            lines = f.readlines()
                        matched = [line.strip() for line in lines if pattern in line]
                        return "\n".join(matched)
                return f"grep: {target_path}: No such file or directory"
        
        # 4. 类 Unix 系统下，直接拉起子进程执行，禁用 shell=True 以确保安全
        result = subprocess.run(
            args,
            capture_output=True,
            text=True,
            encoding='utf-8',
            check=False
        )
        return result.stdout.strip() if result.returncode == 0 else f"CLI 执行失败: {result.stderr.strip()}"
    except Exception as e:
        return f"运行时异常: {str(e)}"

# ============= 🧠 TODO 任务 2：补全 CLI 的 Tool Schema =================
cli_tool_schema = [
    {
        "type": "",
        "function": {
            "name": "",
            "description": "",
            "parameters": {
                "type": "",
                "properties": {
                    
                },
                "required": []
            }
        }
    }
]
# =======================================================================

def run_cli_experiment(target_filename):
    system_prompt = (
        "你是一个文件搜索专员，你拥有通过 run_command 调用系统命令的能力。\n"
        "安全白名单内只允许调用: pwd, ls, grep, cat, echo。\n"
        "你需要利用这些命令，在 data/fc_test 目录下查找目标文件并读取其内容。\n"
        "你可以自由组合和设计命令，比如利用 ls 的递归或 grep 检索等，尽快找到目标文件。"
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"请在 data/fc_test 中查找名为 '{target_filename}' 的文件内容。"}
    ]
    p_tokens, c_tokens, turns = 0, 0, 0
    start_time = time.time()
    
    while turns < 20:
        response = openai_client.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=cli_tool_schema,
            tool_choice="auto"
        )
        p_tokens += response.usage.prompt_tokens
        c_tokens += response.usage.completion_tokens
        turns += 1
        
        assistant_msg = response.choices[0].message
        messages.append(assistant_msg)
        
        if assistant_msg.tool_calls:
            for call in assistant_msg.tool_calls:
                func_name = call.function.name
                func_args = json.loads(call.function.arguments)
                
                if func_name == "run_command":
                    cli_result = run_command(func_args.get("command"))
                else:
                    cli_result = f"Unknown tool: {func_name}"
                    
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "name": func_name,
                    "content": cli_result
                })
        else:
            break
            
    elapsed = time.time() - start_time
    
    import re
    match = re.search(r'level_(\d+)', target_filename)
    depth = int(match.group(1)) if match else None
    expected_key = f"LEVEL_{depth}_SECRET_KEY_9988" if depth is not None else ""
    
    found_key = False
    for msg in messages:
        content = ""
        if hasattr(msg, 'content') and msg.content:
            content = msg.content
        elif isinstance(msg, dict):
            content = msg.get('content', '')
        
        if content and expected_key in content:
            found_key = True
            break
            
    return {
        "time": elapsed,
        "turns": turns,
        "prompt_tokens": p_tokens,
        "completion_tokens": c_tokens,
        "total_tokens": p_tokens + c_tokens,
        "success": found_key,
        "key": expected_key if found_key else "未找到"
    }

print("✅ CLI 实验模块加载成功。")

#### 💻 运行 CLI 命令行检索评测测试

执行深度 1 至 10 的 CLI 命令行检索文件查找，记录并存储每一步的耗时、轮数和 Token 消耗。


In [ ]:
cli_results = []
print("🚀 开始进行 CLI 命令行检索实验 (深度 1 至 10)...")
for depth in range(1, 11):
    target_name, target_path = level_files[depth]
    print(f"   [CLI 测试] 正在查找深度 {depth}: {target_name}...")
    res = run_cli_experiment(target_name)
    cli_results.append({
        "depth": depth,
        "time": res["time"],
        "turns": res["turns"],
        "tokens": res["total_tokens"]
    })
    print(f"             耗时: {res['time']:.2f}s | 轮数: {res['turns']} | 总Token: {res['total_tokens']} | 结果: {res.get('key', '未找到')}")
print("✅ CLI 命令行检索实验测试完成！")

### 🌐 6. 范式三：MCP 细粒度检索实验

通过 Stdio 连接标准的 MCP 文件服务器（`@modelcontextprotocol/server-filesystem`）。大模型同样使用细粒度接口逐层下探。本节展示了如何动态获取 MCP 提供的工具并转化为 OpenAI 工具格式进行调用。


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

def convert_mcp_to_openai_tools(mcp_tools) -> list:
    openai_tools = []
    for tool in mcp_tools.tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.inputSchema
            }
        })
    return openai_tools

async def run_mcp_experiment(target_filename):
    command = "npx.cmd" if os.name == 'nt' else "npx"
    allowed_dir = os.path.abspath("data/fc_test")
    
    server_params = StdioServerParameters(
        command=command,
        args=["-y", "@modelcontextprotocol/server-filesystem", allowed_dir],
        env=None
    )
    
    system_prompt = (
        "你是一个深层目录文件搜索专员。\n"
        "你需要逐步查找目标文件，注意你只能使用 list_directory 一级一级探索，不能直接猜测或递归。\n"
        "请每次使用绝对路径列出当前目录。如果找到目标文件，请使用 read_text_file 读取内容。\n"
        "如果看到子目录，就立刻进入下一级目录。请一步步探索直至找到目标文件。"
    )
    
    mcp_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"请在目录 '{allowed_dir}' 的目录树中，一步步搜寻文件 '{target_filename}' 的具体内容。第一步请先列出 '{allowed_dir}'。"}
    ]
    
    p_tokens, c_tokens, turns = 0, 0, 0
    start_time = time.time()
    
    # 显式传递 errlog=sys.__stderr__ 防止 Jupyter Win32 下 fileno 报错
    async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # 获取 MCP 工具并转化
            
            tools_res = await session.list_tools()
            openai_tools = convert_mcp_to_openai_tools(tools_res)
            
            while turns < 20:
                # ================ 🧠 TODO 任务 3：补全 MCP 工具调用 ==================
                response = openai_client.chat.completions.create(

                )
                # ====================================================================
                
                p_tokens += response.usage.prompt_tokens
                c_tokens += response.usage.completion_tokens
                turns += 1
                
                assistant_msg = response.choices[0].message
                mcp_messages.append(assistant_msg)
                
                if assistant_msg.tool_calls:
                    for call in assistant_msg.tool_calls:
                        func_name = call.function.name
                        func_args = json.loads(call.function.arguments)
                        
                        # 转发调用给后台 MCP 服务器进程
                        mcp_res = await session.call_tool(func_name, arguments=func_args)
                        mcp_out = mcp_res.content[0].text
                        
                        # ============ 🧠 TODO 任务 3：将工具结果加入mcp_message ==============
                        mcp_messages.append({
                            "role": "",
                            "tool_call_id": ,
                            "name": ,
                            "content": 
                        })
                        # ====================================================================
                else:
                    break
                    
    elapsed = time.time() - start_time
    
    import re
    match = re.search(r'level_(\d+)', target_filename)
    depth = int(match.group(1)) if match else None
    expected_key = f"LEVEL_{depth}_SECRET_KEY_9988" if depth is not None else ""
    
    found_key = False
    for msg in mcp_messages:
        content = ""
        if hasattr(msg, 'content') and msg.content:
            content = msg.content
        elif isinstance(msg, dict):
            content = msg.get('content', '')
        
        if content and expected_key in content:
            found_key = True
            break
            
    return {
        "time": elapsed,
        "turns": turns,
        "prompt_tokens": p_tokens,
        "completion_tokens": c_tokens,
        "total_tokens": p_tokens + c_tokens,
        "success": found_key,
        "key": expected_key if found_key else "未找到"
    }
print("✅ MCP 实验模块加载成功。")

#### 🌐 运行 MCP 细粒度检索评测测试

执行深度 1 至 10 的 MCP 细粒度文件查找，记录并存储每一步的耗时、轮数和 Token 消耗。


In [ ]:
mcp_results = []
print("🚀 开始进行 MCP 细粒度检索实验 (深度 1 至 10)...")
for depth in range(1, 11):
    target_name, target_path = level_files[depth]
    print(f"   [MCP 测试] 正在查找深度 {depth}: {target_name}...")
    res = run_async(run_mcp_experiment(target_name))
    mcp_results.append({
        "depth": depth,
        "time": res["time"],
        "turns": res["turns"],
        "tokens": res["total_tokens"]
    })
    print(f"             耗时: {res['time']:.2f}s | 轮数: {res['turns']} | 总Token: {res['total_tokens']} | 结果: {res.get('key', '未找到')}")
print("✅ MCP 细粒度检索实验测试完成！")

### 👥 7. 范式四：层级代理 (Hierarchical Agent) 实验

设计一个经理 (Manager) - 专员 (Worker) 的层级智能体架构。**项目经理**负责顶层调度与结果总结，具体的繁重文件寻找工作则外包给**专员**。专员拥有细粒度工具，逐级下探查找。此架构旨在实现职责分离与上下文隔离。


In [ ]:
def run_agent_experiment(target_filename):
    # 专员 (Worker) 数据统计变量
    worker_metrics = {"turns": 0, "prompt_tokens": 0, "completion_tokens": 0, "time": 0.0}
    
    def ask_file_agent_tracked(query: str) -> str:
        w_start = time.time()
        worker_system_prompt = (
            "你是一个专门在本地 data/fc_test 目录下寻找文件的辅助智能体。\n"
            "你拥有 list_directory 和 read_file 两个文件操作工具。\n"
            "你应该先调用列出文件工具，确认目标是否存在，然后再读取其具体内容。\n"
            "任务完成后，只需给出简洁的内容报告，不要带有多余的问候语或解释。"
        )
        
        worker_messages = [
            {"role": "system", "content": worker_system_prompt},
            {"role": "user", "content": query}
        ]
        
        turn = 0
        while turn < 20: # 专员工具调用限制最大 20 轮
            response = openai_client.chat.completions.create(
                model=model_name,
                messages=worker_messages,
                tools=fine_grained_tools,
                tool_choice="auto"
            )
            
            worker_metrics["prompt_tokens"] += response.usage.prompt_tokens
            worker_metrics["completion_tokens"] += response.usage.completion_tokens
            worker_metrics["turns"] += 1
            turn += 1
            
            assistant_msg = response.choices[0].message
            worker_messages.append(assistant_msg)
            tool_calls = assistant_msg.tool_calls
            
            if not tool_calls:
                worker_metrics["time"] += (time.time() - w_start)
                return assistant_msg.content.strip()
                
            for tool_call in tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)
                
                if func_name == "list_directory":
                    res = str(list_directory(func_args.get("path")))
                elif func_name == "read_file":
                    res = read_file(func_args.get("path"))
                else:
                    res = "Unknown tool"
                    
                worker_messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": func_name,
                    "content": res
                })
                
        worker_metrics["time"] += (time.time() - w_start)
        return "Error: Worker reached maximum turns"

    # 经理 (Manager) 数据统计与调用

    # ============ 🧠 TODO 任务 4：补全 “文件检索专员” 的 Tool Schema ==============
    manager_tools = [
        {
            "type": "",
            "function": {
                "name": "",
                "description": "",
                "parameters": {
                    "type": "",
                    "properties": {
                        
                    },
                    "required": []
                }
            }
        }
    ]
    # ==============================================================================
    
    manager_messages = [
        {"role": "user", "content": f"请帮我查找 data/fc_test 目录下的 {target_filename} 文件。如果存在，请读取其内容并告诉我。"}
    ]
    
    manager_metrics = {"turns": 0, "prompt_tokens": 0, "completion_tokens": 0, "time": 0.0}
    m_start = time.time()
    
    # 经理 Call 1
    response = openai_client.chat.completions.create(
        model=model_name,
        messages=manager_messages,
        tools=manager_tools,
        tool_choice="auto"
    )
    manager_metrics["prompt_tokens"] += response.usage.prompt_tokens
    manager_metrics["completion_tokens"] += response.usage.completion_tokens
    manager_metrics["turns"] += 1
    
    assistant_msg = response.choices[0].message
    manager_messages.append(assistant_msg)
    
    if assistant_msg.tool_calls:
        call = assistant_msg.tool_calls[0]
        args = json.loads(call.function.arguments)
        
        # 呼叫专员执行文件查找（数据已在专员闭包中统计）
        worker_reply = ask_file_agent_tracked(args.get("query"))
        
        manager_messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "name": "ask_file_agent",
            "content": worker_reply
        })
        
        # 经理 Call 2：给出最终结果
        final_response = openai_client.chat.completions.create(
            model=model_name,
            messages=manager_messages
        )
        manager_metrics["prompt_tokens"] += final_response.usage.prompt_tokens
        manager_metrics["completion_tokens"] += final_response.usage.completion_tokens
        manager_metrics["turns"] += 1
        manager_final_msg = final_response.choices[0].message.content
        
    manager_metrics["time"] = time.time() - m_start
    
    import re
    match = re.search(r'level_(\d+)', target_filename)
    depth = int(match.group(1)) if match else None
    expected_key = f"LEVEL_{depth}_SECRET_KEY_9988" if depth is not None else ""
    
    found_key = False
    all_msgs = list(manager_messages)
    if 'manager_final_msg' in locals() and manager_final_msg:
        all_msgs.append({"role": "assistant", "content": manager_final_msg})
        
    for msg in all_msgs:
        content = ""
        if hasattr(msg, 'content') and msg.content:
            content = msg.content
        elif isinstance(msg, dict):
            content = msg.get('content', '')
        
        if content and expected_key in content:
            found_key = True
            break
            
    manager_metrics["success"] = found_key
    manager_metrics["key"] = expected_key if found_key else "未找到"
    return manager_metrics, worker_metrics

print("✅ 层级代理 (Hierarchical Agent) 实验模块加载成功。")

#### 👥 运行层级代理 (Hierarchical Agent) 评测测试

执行深度 1 至 10 的层级代理文件查找，分别记录并存储经理与专员每一步的耗时、轮数和 Token 消耗。


In [ ]:
agent_results = []
print("🚀 开始进行层级代理 (Hierarchical Agent) 实验 (深度 1 至 10)...")
for depth in range(1, 11):
    target_name, target_path = level_files[depth]
    print(f"   [Agent 测试] 正在查找深度 {depth}: {target_name}...")
    manager_res, worker_res = run_agent_experiment(target_name)
    agent_results.append({
        "depth": depth,
        "manager_time": manager_res["time"],
        "manager_turns": manager_res["turns"],
        "manager_tokens": manager_res["prompt_tokens"] + manager_res["completion_tokens"],
        "worker_time": worker_res["time"],
        "worker_turns": worker_res["turns"],
        "worker_tokens": worker_res["prompt_tokens"] + worker_res["completion_tokens"]
    })
    print(f"               经理耗时: {manager_res['time']:.2f}s | 专员调用轮数: {worker_res['turns']} | 总Token: {(manager_res['prompt_tokens']+manager_res['completion_tokens'])+(worker_res['prompt_tokens']+worker_res['completion_tokens'])} | 结果: {manager_res.get('key', '未找到')}")
print("✅ 层级代理 (Hierarchical Agent) 实验测试完成！")

### 📊 8. 汇总实验结果

将每个范式在前面步骤中单独运行产生的测试结果数据合并到一个统一的结果列表中，为后续的数据分析和绘图做好准备。


In [ ]:
results = []
for i in range(10):
    depth = i + 1
    results.append({
        "depth": depth,
        "fc_time": fc_results[i]["time"],
        "fc_turns": fc_results[i]["turns"],
        "fc_tokens": fc_results[i]["tokens"],
        
        "cli_time": cli_results[i]["time"],
        "cli_turns": cli_results[i]["turns"],
        "cli_tokens": cli_results[i]["tokens"],
        
        "mcp_time": mcp_results[i]["time"],
        "mcp_turns": mcp_results[i]["turns"],
        "mcp_tokens": mcp_results[i]["tokens"],
        
        "agent_manager_time": agent_results[i]["manager_time"],
        "agent_manager_turns": agent_results[i]["manager_turns"],
        "agent_manager_tokens": agent_results[i]["manager_tokens"],
        
        "agent_worker_time": agent_results[i]["worker_time"],
        "agent_worker_turns": agent_results[i]["worker_turns"],
        "agent_worker_tokens": agent_results[i]["worker_tokens"],
    })

print("✅ 所有实验结果汇总完毕，数据已准备就绪！")

### 📈 9. 实验结果可视化分析

使用 Pandas 整理汇总好的数据，并利用 Matplotlib 绘制折线图，直观对比四种范式在不同目录深度下的**响应时间**、**工具调用轮数**以及 **Token 消耗量**的变化趋势。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 支持中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

df = pd.DataFrame(results)
display(df)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 响应时间随深度变化曲线
axes[0, 0].plot(df["depth"], df["cli_time"], marker='o', label="CLI 粗粒度")
axes[0, 0].plot(df["depth"], df["fc_time"], marker='s', label="Function Calling 细粒度")
axes[0, 0].plot(df["depth"], df["mcp_time"], marker='^', label="MCP 细粒度")
axes[0, 0].plot(df["depth"], df["agent_manager_time"], marker='x', label="Agent (经理总耗时)")
axes[0, 0].set_title("响应时间 (秒) 随目录深度变化")
axes[0, 0].set_xlabel("目录树深度")
axes[0, 0].set_ylabel("时间 (秒)")
axes[0, 0].grid(True)
axes[0, 0].legend()

# 2. 工具调用轮数随深度变化曲线
axes[0, 1].plot(df["depth"], df["cli_turns"], marker='o', label="CLI 粗粒度")
axes[0, 1].plot(df["depth"], df["fc_turns"], marker='s', label="Function Calling 细粒度")
axes[0, 1].plot(df["depth"], df["mcp_turns"], marker='^', label="MCP 细粒度")
axes[0, 1].plot(df["depth"], df["agent_worker_turns"], marker='d', label="Agent 专员工具轮数")
axes[0, 1].set_title("工具调用轮数 (Turns) 随目录深度变化")
axes[0, 1].set_xlabel("目录树深度")
axes[0, 1].set_ylabel("调用轮数 (Turns)")
axes[0, 1].grid(True)
axes[0, 1].legend()

# 3. Token 消耗总量随深度变化曲线
axes[1, 0].plot(df["depth"], df["cli_tokens"], marker='o', label="CLI 粗粒度")
axes[1, 0].plot(df["depth"], df["fc_tokens"], marker='s', label="Function Calling 细粒度")
axes[1, 0].plot(df["depth"], df["mcp_tokens"], marker='^', label="MCP 细粒度")
axes[1, 0].plot(df["depth"], df["agent_manager_tokens"] + df["agent_worker_tokens"], marker='x', label="Agent 总Token(经理+专员)")
axes[1, 0].set_title("总 Token 消耗随目录深度变化")
axes[1, 0].set_xlabel("目录树深度")
axes[1, 0].set_ylabel("Token 数量")
axes[1, 0].grid(True)
axes[1, 0].legend()

# 4. Agent 层级代理内部 Token 消耗对比（经理 vs 专员）
axes[1, 1].plot(df["depth"], df["agent_manager_tokens"], marker='o', color='purple', label="经理 (Manager) Token")
axes[1, 1].plot(df["depth"], df["agent_worker_tokens"], marker='s', color='orange', label="专员 (Worker) Token")
axes[1, 1].set_title("Layered Agent 内部 Token 分配对比")
axes[1, 1].set_xlabel("目录树深度")
axes[1, 1].set_ylabel("Token 数量")
axes[1, 1].grid(True)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 📈 课后知识复盘与思考练习

### 🏁 实验结论分析
从可视化图表中，我们可以看出这四种范式极具对比性的特征表现：
1. **CLI 递归查找（粗粒度工具）**：其响应时间、工具调用轮数和 Token 消耗基本上是一条**水平的常数直线**（与目录深度无关）。这是因为遍历的工作被完全交给了宿主系统底层的 Python 子进程，大模型只需发出指令和汇总，效率极高且成本极低。
2. **Function Calling 与 MCP（细粒度 ReAct）**：随着目录深度增加，其响应时间与工具调用轮数呈**线性（Linear）**上升；而由于在多轮 ReAct 会话中，前面的所有工具反馈（目录文件列表）都会作为不断增长的上下文在每一次网络请求中重复打包发送，所以其 Token 消耗呈现明显的**二次方级（Quadratic）**增长！这不仅带来了不可忽略的经济开销，也拉长了系统响应的卡顿感。
3. **层级代理 (Hierarchical Agent)**：它完美地展现了职责的分离。负责顶层和用户的**经理 (Manager)** 的工具调用轮数与 Token 开销在各深度下都保持在极低的水平（因为他只发起了 2 次 LLM 调用，一次派发工作，一次翻译总结）。而复杂的底层查找开销，则被完全转移到了**专员 (Worker)** 的上下文当中。虽然专员的 Token 依然在以二次方增长，但**经理 (Manager) 免受了这一爆炸性上下文的冲刷**，这也正是层级代理在设计大型复杂智能体系统时用来“隔离上下文”与“职责专门化”的精髓所在。

### 🧠 思考练习题
1. 在什么业务场景下，我们会**必须**采用细粒度的 MCP 或 Function Calling 工具，而不能粗暴地使用 CLI 来替代它？
2. 在层级代理架构中，如何更好地降低专员 (Worker) 的多轮 ReAct 交互所带来的 Token 二次方开销？（提示：思考是否可以在专员完成工作后及时截断、清理上下文，或者只将精简的最终字符串汇报给经理？）